# TrafficGuard AI
## Phase 5: Helmet Detection Model Training

### Objectives

- Prepare the Google Colab GPU environment.
- Load the validated helmet detection dataset.
- Improve validation source diversity.
- Implement transfer learning using YOLOv8.
- Train a baseline YOLOv8s detection model.
- Train an improved YOLOv8m detection model.
- Analyze training and validation performance.
- Compare model architectures.
- Save model checkpoints and experimental results.

### Target Classes

- Class 0: with_helmet
- Class 1: without_helmet

The independent test dataset remains reserved for
final model evaluation.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import torch

print("=" * 60)
print("GPU ENVIRONMENT VERIFICATION")
print("=" * 60)

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)

if torch.cuda.is_available():

    print("GPU Name:", torch.cuda.get_device_name(0))

    properties = torch.cuda.get_device_properties(0)

    print(
        "GPU Memory:",
        round(properties.total_memory / 1024**3, 2),
        "GB"
    )

else:

    raise RuntimeError(
        "GPU not available. Enable GPU from Runtime settings."
    )

print("\nGPU environment verified successfully!")

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q ultralytics

In [ ]:
import os
import gc
import cv2
import yaml
import json
import time
import random
import shutil
import zipfile
import platform
import itertools
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import ultralytics

from pathlib import Path
from PIL import Image
from ultralytics import YOLO

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style="whitegrid")

plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "font.size": 11
})

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("OpenCV:", cv2.__version__)

print("\nAll required libraries imported successfully!")

In [ ]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

ARCHIVE_PATH = (
    PROJECT_ROOT /
    "datasets/processed/archives/helmet_v1.zip"
)

MANIFEST_PATH = (
    PROJECT_ROOT /
    "results/preprocessing/tables/"
    "processed_dataset_manifest.csv"
)

LOCAL_ROOT = Path(
    "/content/trafficguard_training"
)

DATASET_ROOT = LOCAL_ROOT / "helmet"

RUNS_DIR = PROJECT_ROOT / "runs/helmet"

MODELS_DIR = PROJECT_ROOT / "models/helmet"

RESULTS_DIR = PROJECT_ROOT / "results/helmet_training"

FIGURES_DIR = RESULTS_DIR / "figures"

TABLES_DIR = RESULTS_DIR / "tables"

REPORTS_DIR = RESULTS_DIR / "reports"

for folder in [
    LOCAL_ROOT,
    RUNS_DIR,
    MODELS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    REPORTS_DIR
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

assert ARCHIVE_PATH.exists(), "Helmet archive missing!"

assert MANIFEST_PATH.exists(), "Manifest missing!"

print("Dataset archive:", ARCHIVE_PATH)

print("Training runs:", RUNS_DIR)

print("Model storage:", MODELS_DIR)

print("Results storage:", RESULTS_DIR)

print("\nAll project paths configured successfully!")

In [ ]:
LOCAL_ARCHIVE = LOCAL_ROOT / "helmet_v1.zip"

print("Copying dataset archive to local storage...")

shutil.copy2(
    ARCHIVE_PATH,
    LOCAL_ARCHIVE
)

print("Extracting helmet dataset...")

with zipfile.ZipFile(
    LOCAL_ARCHIVE,
    "r"
) as zip_ref:

    zip_ref.extractall(LOCAL_ROOT)

assert DATASET_ROOT.exists()

print("\nDataset extracted successfully!")

print("Dataset location:", DATASET_ROOT)

In [ ]:
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".webp"
}

def count_dataset(root):

    records = []

    for split in ["train", "val", "test"]:

        image_dir = root / split / "images"

        label_dir = root / split / "labels"

        images = [
            p for p in image_dir.iterdir()
            if p.suffix.lower() in IMAGE_EXTENSIONS
        ]

        labels = list(label_dir.glob("*.txt"))

        assert {p.stem for p in images} == {
            p.stem for p in labels
        }

        records.append({
            "Split": split,
            "Images": len(images),
            "Labels": len(labels)
        })

    return pd.DataFrame(records)


dataset_summary = count_dataset(DATASET_ROOT)

display(dataset_summary)

assert dataset_summary["Images"].sum() == 1759

assert (
    dataset_summary["Images"] ==
    dataset_summary["Labels"]
).all()

print("\nDataset extraction verified successfully!")

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)

helmet_manifest = manifest[
    manifest["Dataset"] == "Helmet"
].copy()

assert len(helmet_manifest) == 1759

group_summary = (
    helmet_manifest.groupby(
        ["Final Split", "Group ID"]
    )
    .agg(
        Images=("Final Image", "size"),

        Class0=("Class 0 Objects", "sum"),

        Class1=("Class 1 Objects", "sum")
    )
    .reset_index()
)

display(group_summary)

print("\nIndependent groups by split:")

display(
    helmet_manifest.groupby("Final Split")[
        "Group ID"
    ].nunique()
)

In [ ]:
development = helmet_manifest[
    helmet_manifest["Final Split"].isin(
        ["train", "val"]
    )
].copy()

development_groups = (
    development.groupby("Group ID")
    .agg(
        Images=("Final Image", "size"),

        Class0=("Class 0 Objects", "sum"),

        Class1=("Class 1 Objects", "sum")
    )
)

group_ids = development_groups.index.tolist()

totals = development_groups[
    ["Images", "Class0", "Class1"]
].sum().to_numpy(dtype=float)

best_score = float("inf")

best_val_groups = None

# Target approximately 18% of development images
# for validation.
target_val_ratio = 0.18

for n in range(2, len(group_ids) - 1):

    for combination in itertools.combinations(
        group_ids, n
    ):

        val_totals = (
            development_groups.loc[
                list(combination)
            ][["Images", "Class0", "Class1"]]
            .sum()
            .to_numpy(dtype=float)
        )

        train_totals = totals - val_totals

        if np.any(val_totals == 0):
            continue

        if np.any(train_totals == 0):
            continue

        ratios = val_totals / totals

        if not 0.12 <= ratios[0] <= 0.25:
            continue

        score = (
            3 * (ratios[0] - target_val_ratio)**2

            + np.sum(
                (ratios[1:] - target_val_ratio)**2
            )
        )

        if score < best_score:

            best_score = score

            best_val_groups = set(combination)


assert best_val_groups is not None, (
    "No acceptable split found. Review group sizes."
)

revised_manifest = helmet_manifest.copy()

dev_mask = revised_manifest["Final Split"].isin(
    ["train", "val"]
)

revised_manifest.loc[
    dev_mask,
    "Final Split"
] = revised_manifest.loc[
    dev_mask,
    "Group ID"
].map(
    lambda group:
    "val" if group in best_val_groups else "train"
)

revised_summary = (
    revised_manifest.groupby("Final Split")
    .agg(
        Images=("Final Image", "size"),

        Groups=("Group ID", "nunique"),

        Class0=("Class 0 Objects", "sum"),

        Class1=("Class 1 Objects", "sum")
    )
    .reset_index()
)

display(revised_summary)

print("Selected validation groups:", best_val_groups)

print("Split optimization score:", best_score)

assert revised_summary.loc[
    revised_summary["Final Split"] == "val",
    "Groups"
].iloc[0] >= 2

assert (
    revised_manifest.groupby("Group ID")[
        "Final Split"
    ].nunique() == 1
).all()

# Confirm the original test images remain unchanged.
old_test = set(
    helmet_manifest.loc[
        helmet_manifest["Final Split"] == "test",
        "Final Image"
    ]
)

new_test = set(
    revised_manifest.loc[
        revised_manifest["Final Split"] == "test",
        "Final Image"
    ]
)

assert old_test == new_test

print("\nValidation diversity improved.")

print("Original test set preserved.")

In [ ]:
REVISED_ROOT = LOCAL_ROOT / "helmet_v2"

for split in ["train", "val", "test"]:

    for folder in ["images", "labels"]:

        (
            REVISED_ROOT /
            split /
            folder
        ).mkdir(
            parents=True,
            exist_ok=True
        )

for row in revised_manifest.to_dict("records"):

    new_split = row["Final Split"]

    source_image = (
        LOCAL_ROOT /
        row["Relative Image Path"]
    )

    source_label = (
        LOCAL_ROOT /
        row["Relative Label Path"]
    )

    target_image = (
        REVISED_ROOT /
        new_split /
        "images" /
        source_image.name
    )

    target_label = (
        REVISED_ROOT /
        new_split /
        "labels" /
        source_label.name
    )

    shutil.copy2(
        source_image,
        target_image
    )

    shutil.copy2(
        source_label,
        target_label
    )

revised_counts = count_dataset(REVISED_ROOT)

display(revised_counts)

assert revised_counts["Images"].sum() == 1759

print("\nRevised helmet dataset created successfully!")

In [ ]:
DATA_YAML = REVISED_ROOT / "data.yaml"

dataset_config = {
    "path": str(REVISED_ROOT),

    "train": "train/images",

    "val": "val/images",

    "test": "test/images",

    "nc": 2,

    "names": {
        0: "with_helmet",
        1: "without_helmet"
    }
}

with open(
    DATA_YAML,
    "w",
    encoding="utf-8"
) as file:

    yaml.safe_dump(
        dataset_config,
        file,
        sort_keys=False
    )

print(DATA_YAML.read_text())

In [ ]:
revised_manifest.to_csv(
    TABLES_DIR / "helmet_v2_manifest.csv",
    index=False
)

revised_summary.to_csv(
    TABLES_DIR / "helmet_v2_split_summary.csv",
    index=False
)

V2_ARCHIVE = (
    PROJECT_ROOT /
    "datasets/processed/archives/helmet_v2"
)

archive = shutil.make_archive(
    base_name=str(V2_ARCHIVE),

    format="zip",

    root_dir=str(LOCAL_ROOT),

    base_dir="helmet_v2"
)

print("Revised dataset archive:", archive)

print(
    "Archive size:",
    round(Path(archive).stat().st_size / 1024**2, 2),
    "MB"
)

print("\nRevised dataset saved permanently.")

In [ ]:
environment_info = {
    "Python": platform.python_version(),

    "PyTorch": torch.__version__,

    "Ultralytics": ultralytics.__version__,

    "CUDA": torch.version.cuda,

    "GPU": torch.cuda.get_device_name(0),

    "Random Seed": SEED,

    "Dataset Version": "helmet_v2"
}

environment_path = (
    REPORTS_DIR / "training_environment.json"
)

environment_path.write_text(
    json.dumps(environment_info, indent=4),
    encoding="utf-8"
)

print(
    json.dumps(environment_info, indent=4)
)

print("\nEnvironment information saved.")

In [ ]:
baseline_model = YOLO("yolov8s.pt")

baseline_model.info()

print("\nYOLOv8s pretrained model loaded.")

In [ ]:
BASELINE_NAME = "helmet_yolov8s_baseline_v1"

BASELINE_CONFIG = {

    "data": str(DATA_YAML),

    "epochs": 50,

    "imgsz": 640,

    "batch": 16,

    "device": 0,

    "workers": 2,

    "optimizer": "auto",

    "patience": 15,

    "pretrained": True,

    "amp": True,

    "seed": SEED,

    "deterministic": True,

    "cache": False,

    "val": True,

    "plots": True,

    "save": True,

    "save_period": 10,

    "close_mosaic": 10,

    "project": str(RUNS_DIR),

    "name": BASELINE_NAME,

    "exist_ok": False
}

print("BASELINE TRAINING CONFIGURATION")

print(
    json.dumps(
        BASELINE_CONFIG,
        indent=4
    )
)

config_path = (
    REPORTS_DIR /
    "helmet_baseline_config.json"
)

config_path.write_text(
    json.dumps(BASELINE_CONFIG, indent=4)
)

In [ ]:
gc.collect()

torch.cuda.empty_cache()

EXPECTED_RUN = RUNS_DIR / BASELINE_NAME

assert not EXPECTED_RUN.exists(), (
    "This experiment already exists. "
    "Resume it or use a new experiment name."
)

start_time = time.time()

baseline_results = baseline_model.train(
    **BASELINE_CONFIG
)

baseline_training_time = (
    time.time() - start_time
)

BASELINE_RUN = Path(
    baseline_model.trainer.save_dir
)

print("\n" + "=" * 60)

print("BASELINE TRAINING COMPLETED")

print("=" * 60)

print(
    "Training duration:",
    round(baseline_training_time / 60, 2),
    "minutes"
)

print("Results directory:", BASELINE_RUN)

print(
    "Best weights:",
    BASELINE_RUN / "weights/best.pt"
)

In [ ]:
BASELINE_RUN = Path(
    baseline_model.trainer.save_dir
)

BASELINE_BEST = (
    BASELINE_RUN /
    "weights/best.pt"
)

BASELINE_LAST = (
    BASELINE_RUN /
    "weights/last.pt"
)

assert BASELINE_BEST.exists()

assert BASELINE_LAST.exists()

print("BASELINE TRAINING OUTPUTS")

for file in sorted(BASELINE_RUN.iterdir()):

    print(file.name)

print("\nBest model:", BASELINE_BEST)

print(
    "Best model size:",
    round(
        BASELINE_BEST.stat().st_size / 1024**2,
        2
    ),
    "MB"
)

In [ ]:
baseline_history = pd.read_csv(
    BASELINE_RUN / "results.csv"
)

baseline_history.columns = (
    baseline_history.columns.str.strip()
)

print("Completed epochs:", len(baseline_history))

print("\nAvailable metrics:")

print(
    baseline_history.columns.tolist()
)

display(
    baseline_history.tail(10)
)

baseline_history.to_csv(
    TABLES_DIR /
    "helmet_baseline_training_history.csv",

    index=False
)

In [ ]:
loss_types = [
    "box_loss",
    "cls_loss",
    "dfl_loss"
]

fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 5)
)

for ax, loss in zip(
    axes,
    loss_types
):

    train_column = f"train/{loss}"

    val_column = f"val/{loss}"

    ax.plot(
        baseline_history["epoch"],

        baseline_history[train_column],

        label="Training",

        linewidth=2
    )

    ax.plot(
        baseline_history["epoch"],

        baseline_history[val_column],

        label="Validation",

        linewidth=2
    )

    ax.set_title(
        loss.replace("_", " ").title()
    )

    ax.set_xlabel("Epoch")

    ax.set_ylabel("Loss")

    ax.legend()

plt.suptitle(
    "YOLOv8s Helmet Detection — Learning Curves",
    fontsize=16
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "01_baseline_loss_curves.png",

    dpi=300,

    bbox_inches="tight"
)

plt.show()

In [ ]:
metric_columns = {
    "metrics/precision(B)": "Precision",

    "metrics/recall(B)": "Recall",

    "metrics/mAP50(B)": "mAP@50",

    "metrics/mAP50-95(B)": "mAP@50-95"
}

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 10)
)

for ax, (column, title) in zip(
    axes.flat,

    metric_columns.items()
):

    ax.plot(
        baseline_history["epoch"],

        baseline_history[column] * 100,

        linewidth=2
    )

    ax.set_title(title)

    ax.set_xlabel("Epoch")

    ax.set_ylabel("Performance (%)")

    ax.set_ylim(0, 105)

plt.suptitle(
    "YOLOv8s Helmet Model — Validation Performance",
    fontsize=16
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "02_baseline_performance_curves.png",

    dpi=300,

    bbox_inches="tight"
)

plt.show()

In [ ]:
baseline_eval_model = YOLO(
    str(BASELINE_BEST)
)

baseline_metrics = baseline_eval_model.val(

    data=str(DATA_YAML),

    split="val",

    imgsz=640,

    batch=8,

    device=0,

    conf=0.001,

    iou=0.7,

    plots=True,

    project=str(RESULTS_DIR / "validation"),

    name="yolov8s_baseline",

    exist_ok=True
)

precision = float(
    baseline_metrics.box.mp
)

recall = float(
    baseline_metrics.box.mr
)

map50 = float(
    baseline_metrics.box.map50
)

map50_95 = float(
    baseline_metrics.box.map
)

f1 = (
    2 * precision * recall /
    (precision + recall)
    if precision + recall > 0
    else 0
)

baseline_summary = pd.DataFrame([{

    "Model": "YOLOv8s",

    "Dataset": "Helmet v2",

    "Image Size": 640,

    "Completed Epochs": len(baseline_history),

    "Precision": precision,

    "Recall": recall,

    "F1 from Macro P/R": f1,

    "mAP50": map50,

    "mAP50-95": map50_95,

    "Inference Time (ms)": float(
        baseline_metrics.speed["inference"]
    ),

    "Model Size MB": (
        BASELINE_BEST.stat().st_size / 1024**2
    )

}])

display(baseline_summary.round(4))

baseline_summary.to_csv(
    TABLES_DIR / "helmet_baseline_metrics.csv",

    index=False
)

In [ ]:
def get_class_metrics(metrics, model_name, names):

    records = []

    for index, class_id in enumerate(
        metrics.box.ap_class_index
    ):

        class_id = int(class_id)

        p = float(metrics.box.p[index])

        r = float(metrics.box.r[index])

        f1 = (
            2 * p * r / (p + r)
            if p + r > 0
            else 0
        )

        records.append({

            "Model": model_name,

            "Class ID": class_id,

            "Class": names[class_id],

            "Precision": p,

            "Recall": r,

            "F1": f1,

            "mAP50": float(
                metrics.box.ap50[index]
            ),

            "mAP50-95": float(
                metrics.box.ap[index]
            )
        })

    return pd.DataFrame(records)


baseline_class_metrics = get_class_metrics(

    baseline_metrics,

    "YOLOv8s",

    baseline_eval_model.names
)

display(
    baseline_class_metrics.round(4)
)

baseline_class_metrics.to_csv(
    TABLES_DIR /
    "helmet_baseline_per_class_metrics.csv",

    index=False
)

In [ ]:
from IPython.display import display

graph_names = [
    "results.png",

    "confusion_matrix.png",

    "confusion_matrix_normalized.png",

    "BoxPR_curve.png",

    "BoxF1_curve.png",

    "BoxP_curve.png",

    "BoxR_curve.png"
]

for name in graph_names:

    path = BASELINE_RUN / name

    if path.exists():

        print("\n" + name)

        with Image.open(path) as img:

            display(img.copy())

    else:

        print("Not found:", name)

print("\nAvailable PNG files:")

for path in BASELINE_RUN.glob("*.png"):

    print(path.name)

In [ ]:
val_images = sorted([

    p for p in (
        REVISED_ROOT /
        "val/images"
    ).iterdir()

    if p.suffix.lower() in IMAGE_EXTENSIONS
])

rng = random.Random(SEED)

sample_images = rng.sample(

    val_images,

    min(9, len(val_images))
)

baseline_predictions = (
    baseline_eval_model.predict(

        source=[
            str(path)
            for path in sample_images
        ],

        conf=0.25,

        imgsz=640,

        device=0,

        verbose=False
    )
)

fig, axes = plt.subplots(
    3, 3,
    figsize=(17, 13)
)

for ax in axes.flat:

    ax.axis("off")

for ax, prediction in zip(
    axes.flat,
    baseline_predictions
):

    plotted = prediction.plot()

    ax.imshow(
        cv2.cvtColor(
            plotted,
            cv2.COLOR_BGR2RGB
        )
    )

    ax.set_title(
        "YOLOv8s — Validation Prediction"
    )

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "03_baseline_prediction_samples.png",

    dpi=300,

    bbox_inches="tight"
)

plt.show()

In [ ]:
baseline_destination = (
    MODELS_DIR /
    "helmet_yolov8s_baseline.pt"
)

shutil.copy2(
    BASELINE_BEST,

    baseline_destination
)

print("Baseline model saved permanently.")

print("Location:", baseline_destination)

print(
    "Size:",
    round(
        baseline_destination.stat().st_size /
        1024**2,

        2
    ),

    "MB"
)

In [ ]:
del baseline_results

del baseline_model

del baseline_eval_model

del baseline_predictions

gc.collect()

torch.cuda.empty_cache()

print("GPU memory cleanup completed.")

In [ ]:
improved_model = YOLO("yolov8m.pt")

improved_model.info()

IMPROVED_NAME = (
    "helmet_yolov8m_improved_v1"
)

IMPROVED_CONFIG = {

    "data": str(DATA_YAML),

    "epochs": 100,

    "imgsz": 640,

    "batch": 8,

    "device": 0,

    "workers": 2,

    "optimizer": "AdamW",

    "lr0": 0.001,

    "lrf": 0.01,

    "weight_decay": 0.0005,

    "warmup_epochs": 3.0,

    "cos_lr": True,

    "patience": 20,

    "pretrained": True,

    "amp": True,

    "seed": SEED,

    "deterministic": True,

    "cache": False,

    "val": True,

    "plots": True,

    "save": True,

    "save_period": 10,

    # Data augmentation

    "hsv_h": 0.015,

    "hsv_s": 0.5,

    "hsv_v": 0.35,

    "degrees": 5.0,

    "translate": 0.1,

    "scale": 0.4,

    "shear": 0.0,

    "perspective": 0.0002,

    "flipud": 0.0,

    "fliplr": 0.5,

    "mosaic": 0.7,

    "mixup": 0.0,

    "close_mosaic": 15,

    "project": str(RUNS_DIR),

    "name": IMPROVED_NAME,

    "exist_ok": False
}

print(
    json.dumps(
        IMPROVED_CONFIG,
        indent=4
    )
)

(
    REPORTS_DIR /
    "helmet_improved_config.json"
).write_text(
    json.dumps(IMPROVED_CONFIG, indent=4)
)

In [ ]:
gc.collect()

torch.cuda.empty_cache()

assert not (
    RUNS_DIR / IMPROVED_NAME
).exists(), (
    "Experiment already exists. "
    "Resume it or choose a new name."
)

start_time = time.time()

improved_results = improved_model.train(
    **IMPROVED_CONFIG
)

improved_training_time = (
    time.time() - start_time
)

IMPROVED_RUN = Path(
    improved_model.trainer.save_dir
)

IMPROVED_BEST = (
    IMPROVED_RUN /
    "weights/best.pt"
)

print("\n" + "=" * 60)

print("IMPROVED MODEL TRAINING COMPLETED")

print("=" * 60)

print(
    "Training duration:",
    round(
        improved_training_time / 60,
        2
    ),
    "minutes"
)

print("Run directory:", IMPROVED_RUN)

print("Best model:", IMPROVED_BEST)

In [ ]:
improved_history = pd.read_csv(
    IMPROVED_RUN / "results.csv"
)

improved_history.columns = (
    improved_history.columns.str.strip()
)

print(
    "Completed epochs:",
    len(improved_history)
)

display(
    improved_history.tail(10)
)

improved_history.to_csv(
    TABLES_DIR /
    "helmet_improved_training_history.csv",

    index=False
)

In [ ]:
improved_eval_model = YOLO(
    str(IMPROVED_BEST)
)

improved_metrics = improved_eval_model.val(

    data=str(DATA_YAML),

    split="val",

    imgsz=640,

    batch=8,

    device=0,

    conf=0.001,

    iou=0.7,

    plots=True,

    project=str(RESULTS_DIR / "validation"),

    name="yolov8m_improved",

    exist_ok=True
)

p = float(improved_metrics.box.mp)

r = float(improved_metrics.box.mr)

f1 = (
    2 * p * r / (p + r)
    if p + r > 0
    else 0
)

improved_summary = pd.DataFrame([{

    "Model": "YOLOv8m",

    "Dataset": "Helmet v2",

    "Image Size": 640,

    "Completed Epochs": len(improved_history),

    "Precision": p,

    "Recall": r,

    "F1 from Macro P/R": f1,

    "mAP50": float(
        improved_metrics.box.map50
    ),

    "mAP50-95": float(
        improved_metrics.box.map
    ),

    "Inference Time (ms)": float(
        improved_metrics.speed["inference"]
    ),

    "Model Size MB": (
        IMPROVED_BEST.stat().st_size / 1024**2
    )

}])

display(improved_summary.round(4))

improved_summary.to_csv(
    TABLES_DIR /
    "helmet_improved_metrics.csv",

    index=False
)

In [ ]:
improved_class_metrics = get_class_metrics(

    improved_metrics,

    "YOLOv8m",

    improved_eval_model.names
)

display(
    improved_class_metrics.round(4)
)

improved_class_metrics.to_csv(
    TABLES_DIR /
    "helmet_improved_per_class_metrics.csv",

    index=False
)

In [ ]:
improved_destination = (
    MODELS_DIR /
    "helmet_yolov8m_improved.pt"
)

shutil.copy2(

    IMPROVED_BEST,

    improved_destination
)

print("Improved model saved permanently.")

print("Location:", improved_destination)

print(
    "Model size:",

    round(
        improved_destination.stat().st_size /
        1024**2,

        2
    ),

    "MB"
)

In [ ]:
baseline_summary = pd.read_csv(
    TABLES_DIR / "helmet_baseline_metrics.csv"
)

improved_summary = pd.read_csv(
    TABLES_DIR / "helmet_improved_metrics.csv"
)

comparison_df = pd.concat(

    [
        baseline_summary,

        improved_summary
    ],

    ignore_index=True
)

display_df = comparison_df.copy()

metric_names = [

    "Precision",

    "Recall",

    "F1 from Macro P/R",

    "mAP50",

    "mAP50-95"
]

display_df[metric_names] = (
    display_df[metric_names] * 100
).round(2)

display(display_df.T)

comparison_df.to_csv(
    TABLES_DIR /
    "helmet_model_comparison.csv",

    index=False
)

In [ ]:
plot_data = comparison_df.melt(

    id_vars=["Model"],

    value_vars=[
        "Precision",

        "Recall",

        "F1 from Macro P/R",

        "mAP50",

        "mAP50-95"
    ],

    var_name="Metric",

    value_name="Score"
)

plot_data["Score"] *= 100

plt.figure(figsize=(13, 7))

ax = sns.barplot(

    data=plot_data,

    x="Metric",

    y="Score",

    hue="Model",

    palette=[
        "#2563eb",

        "#16a34a"
    ]
)

for container in ax.containers:

    ax.bar_label(

        container,

        fmt="%.2f",

        padding=3,

        fontsize=9
    )

plt.ylim(0, 110)

plt.title(
    "Helmet Detection Model Performance Comparison"
)

plt.ylabel("Validation Performance (%)")

plt.xlabel("Evaluation Metric")

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "04_helmet_model_comparison.png",

    dpi=300,

    bbox_inches="tight"
)

plt.show()

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 6)
)

history_sources = [
    (
        baseline_history,
        "YOLOv8s"
    ),

    (
        improved_history,
        "YOLOv8m"
    )
]

for history, name in history_sources:

    axes[0].plot(

        history["epoch"],

        history["metrics/mAP50(B)"] * 100,

        label=name,

        linewidth=2
    )

    axes[1].plot(

        history["epoch"],

        history["metrics/mAP50-95(B)"] * 100,

        label=name,

        linewidth=2
    )

axes[0].set_title("mAP@50 Progress")

axes[1].set_title("mAP@50–95 Progress")

for ax in axes:

    ax.set_xlabel("Epoch")

    ax.set_ylabel("Validation Performance (%)")

    ax.set_ylim(0, 105)

    ax.legend()

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "05_helmet_learning_curve_comparison.png",

    dpi=300,

    bbox_inches="tight"
)

plt.show()

In [ ]:
baseline_class_metrics = pd.read_csv(
    TABLES_DIR /
    "helmet_baseline_per_class_metrics.csv"
)

improved_class_metrics = pd.read_csv(
    TABLES_DIR /
    "helmet_improved_per_class_metrics.csv"
)

class_comparison = pd.concat(

    [
        baseline_class_metrics,

        improved_class_metrics
    ],

    ignore_index=True
)

display(
    class_comparison.round(4)
)

class_comparison.to_csv(
    TABLES_DIR /
    "helmet_per_class_comparison.csv",

    index=False
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 6)
)

for ax, metric in zip(
    axes,
    ["Recall", "mAP50"]
):

    sns.barplot(

        data=class_comparison,

        x="Class",

        y=metric,

        hue="Model",

        ax=ax
    )

    ax.set_title(
        f"Helmet Class-wise {metric}"
    )

    ax.set_ylim(0, 1.1)

    for container in ax.containers:

        ax.bar_label(
            container,
            fmt="%.3f",
            padding=3
        )

plt.tight_layout()

plt.savefig(
    FIGURES_DIR /
    "06_helmet_classwise_comparison.png",

    dpi=300,

    bbox_inches="tight"
)

plt.show()

In [ ]:
violation_metrics = class_comparison[
    class_comparison["Class"] == "without_helmet"
][
    ["Model", "Recall"]
].rename(
    columns={
        "Recall": "Violation Recall"
    }
)

selection_df = comparison_df.merge(

    violation_metrics,

    on="Model"
)

assert len(selection_df) == len(comparison_df)

selection_df["Selection Score"] = (

    0.40 * selection_df["mAP50-95"]

    + 0.25 * selection_df["mAP50"]

    + 0.25 * selection_df["Violation Recall"]

    + 0.10 * selection_df["Precision"]
)

selection_df = selection_df.sort_values(

    "Selection Score",

    ascending=False
)

display(
    selection_df[
        [
            "Model",

            "mAP50",

            "mAP50-95",

            "Violation Recall",

            "Precision",

            "Selection Score"
        ]
    ].round(4)
)

candidate_name = selection_df.iloc[0]["Model"]

print(
    "\nPreliminary selected model:",
    candidate_name
)

selection_df.to_csv(
    TABLES_DIR /
    "helmet_model_selection.csv",

    index=False
)

In [ ]:
model_paths = {

    "YOLOv8s": (
        MODELS_DIR /
        "helmet_yolov8s_baseline.pt"
    ),

    "YOLOv8m": (
        MODELS_DIR /
        "helmet_yolov8m_improved.pt"
    )
}

selected_path = model_paths[
    candidate_name
]

candidate_path = (
    MODELS_DIR /
    "helmet_candidate_best.pt"
)

shutil.copy2(
    selected_path,

    candidate_path
)

print("Preliminary best model saved.")

print("Selected architecture:", candidate_name)

print("Model path:", candidate_path)

In [ ]:
report_lines = [

    "TRAFFICGUARD AI",

    "PHASE 5 - HELMET MODEL TRAINING REPORT",

    "=" * 65,

    "",

    "TRAINING ENVIRONMENT",

    "-" * 65,

    json.dumps(
        environment_info,
        indent=4
    ),

    "",

    "REVISED DATASET DISTRIBUTION",

    "-" * 65,

    revised_summary.to_string(
        index=False
    ),

    "",

    "MODEL PERFORMANCE COMPARISON",

    "-" * 65,

    comparison_df.round(4).to_string(
        index=False
    ),

    "",

    "CLASS-WISE PERFORMANCE",

    "-" * 65,

    class_comparison.round(4).to_string(
        index=False
    ),

    "",

    "MODEL SELECTION",

    "-" * 65,

    selection_df.round(4).to_string(
        index=False
    ),

    "",

    f"Preliminary best model: {candidate_name}",

    f"Candidate checkpoint: {candidate_path}",

    "",

    "EXPERIMENTAL NOTES",

    "-" * 65,

    "Both models used pretrained YOLOv8 weights.",

    "Both models used the helmet_v2 dataset.",

    "Validation contains multiple source groups.",

    "The original independent test set was preserved.",

    "Training-only augmentation was used.",

    "Model selection used validation metrics only.",

    "Final test evaluation has not been performed.",

    "",

    "LIMITATIONS",

    "-" * 65,

    "Only 13 identified helmet video sources are available.",

    "Training data contains moderate class imbalance.",

    "Unknown source relationships may still exist.",

    "Real-world generalization requires further evaluation.",

    "",

    "NEXT PHASE",

    "-" * 65,

    "Seatbelt detection model training."
]

report_text = "\n".join(report_lines)

report_path = (
    REPORTS_DIR /
    "helmet_training_report.txt"
)

report_path.write_text(

    report_text,

    encoding="utf-8"
)

print(report_text)

print("\nReport saved at:", report_path)

In [ ]:
print("=" * 65)

print("TRAFFICGUARD AI - PHASE 5 OUTPUTS")

print("=" * 65)

for folder in [

    MODELS_DIR,

    FIGURES_DIR,

    TABLES_DIR,

    REPORTS_DIR
]:

    print("\n", folder.name.upper())

    files = sorted([

        p for p in folder.iterdir()

        if p.is_file()
    ])

    for file in files:

        print(

            f" - {file.name} "

            f"({file.stat().st_size / 1024:.1f} KB)"
        )

    print("Total files:", len(files))

assert (
    MODELS_DIR /
    "helmet_yolov8s_baseline.pt"
).exists()

assert (
    MODELS_DIR /
    "helmet_yolov8m_improved.pt"
).exists()

assert candidate_path.exists()

assert report_path.exists()

print("\n" + "=" * 65)

print("PHASE 5 TRAINING ARTIFACTS VERIFIED")

print("Final performance review required.")

print("Next phase: Seatbelt model training.")

print("=" * 65)